In [11]:
import pyodbc

print(pyodbc.drivers())

['SQL Server', 'ODBC Driver 17 for SQL Server', 'SQL Server Native Client 11.0', 'SQL Server Native Client RDA 11.0', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']


##LocalDB

In [6]:
import pyodbc
import pandas as pd

conn = pyodbc.connect(
    r"DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=TWTPENO251004;"
    r"DATABASE=Optmizer;"
    r"Trusted_Connection=yes;"
    r"TrustServerCertificate=yes;"
)

print("SQL Server 連線成功")


SQL Server 連線成功


In [7]:
sql = """
SELECT TOP 100 *
FROM Customer
"""

df = pd.read_sql(sql, conn)

df.head()

C:\Users\harry\AppData\Local\Temp\ipykernel_12464\236354777.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


,First_Name,Last_Name,Address,City,Country,Birth_Date
0,James ...,Smith ...,123 Main St ...,New York ...,USA,1988-03-12
1,Emma ...,Johnson ...,45 Park Ave ...,Los Angeles ...,USA,1992-07-25
2,Liam ...,Williams ...,78 King Rd ...,Chicago ...,USA,1985-11-04
3,Olivia ...,Brown ...,9 Sunset Blvd ...,Houston ...,USA,1995-02-18
4,Noah ...,Jones ...,66 Ocean Dr ...,Miami ...,USA,1990-09-30


In [13]:
df_Customer = pd.DataFrame({
    "First_Name": ["James", "Emily", "Michael", "Sophia", "Daniel"],
    "Last_Name": ["Smith", "Johnson", "Brown", "Wilson", "Lee"],
    "Address": [
        "123 Main St",
        "456 Oak Ave",
        "789 Pine Rd",
        "321 Maple St",
        "567 King St"
    ],
    "City": [
        "New York",
        "Los Angeles",
        "Chicago",
        "Toronto",
        "London"
    ],
    "Country": [
        "USA",
        "USA",
        "USA",
        "Canada",
        "UK"
    ],
    "Birth_Date": [
        "1988-03-12",
        "1992-07-25",
        "1985-11-08",
        "1995-04-16",
        "1990-09-21"
    ]
})

In [14]:
import pandas as pd

def insert_customer(conn, df):
    cursor = conn.cursor()

    sql = """
    INSERT INTO Customer (
        First_Name,
        Last_Name,
        Address,
        City,
        Country,
        Birth_Date
    )
    VALUES (?, ?, ?, ?, ?, ?)
    """

    # DataFrame → list of tuples
    data = list(
        df[
            [
                "First_Name",
                "Last_Name",
                "Address",
                "City",
                "Country",
                "Birth_Date"
            ]
        ].itertuples(index=False, name=None)
    )

    try:
        cursor.fast_executemany = True
        cursor.executemany(sql, data)

        conn.commit()

        print(f"成功寫入 {len(data)} 筆資料")

    except Exception as e:
        conn.rollback()
        print("寫入失敗：", e)

    finally:
        cursor.close()

In [ ]:
#format date
df_Customer["Birth_Date"] = pd.to_datetime(df_Customer["Birth_Date"]).dt.date

In [15]:
insert_customer(conn, df_Customer)

成功寫入 5 筆資料


In [ ]:
def insert_data(df):
    # conn = get_connection()

    try:
        cursor = conn.cursor()
        for index, row in df.iterrows():
            cursor.execute()
        conn.commit()
    finally:
        conn.close()

#FubonArtDB

In [5]:
import pymssql
import pandas as pd

def get_connection():
    return pymssql.connect(
        server="10.140.10.3",
        user="utikirw",
        password="utikirwp@w",
        database="FubonArtDB",
        charset="utf8"
    )


def load_data():
    conn = get_connection()

    sql = """
    SELECT *
    FROM Orders
    """

    try:
        df = pd.read_sql(sql, conn)
        return df
    finally:
        conn.close()


df = load_data()

print(df.head(3))
print(f"資料筆數：{len(df):,}")

       PKNO     order_number order_user_id          order_datetime  \
0  B1Z3SRLE  FB2103267001101      N1NEVD8L 2023-03-26 14:30:14.740   
1  B1Z43LVB  FB2103267001401      N1NEVD8L 2023-03-26 16:02:42.660   
2  B1Z43LVL  FB2103267001402      N1NEVD8L 2023-03-26 16:08:02.700   

  order_site_id  pay_method get_method order_type shopper  \
0             1           4       None          1      64   
1             1           1       None          1     128   
2             1           1       None          1     128   

                                 shopper_phone  ...  InvoiceType  barcode  \
0  Vgs5yrtL+kVCnQcstURUdfSs9rYA/eWurHRdk6NbURc  ...          NaN      NaN   
1  Vgs5yrtL+kVCnQcstURUdfSs9rYA/eWurHRdk6NbURc  ...          NaN      NaN   
2  Vgs5yrtL+kVCnQcstURUdfSs9rYA/eWurHRdk6NbURc  ...          NaN      NaN   

  tax_id LoveCode mail  Is_ActivitiesGroup  Organization  Dep Address  \
0   None     None  NaN                 NaN           NaN  NaN     NaN   
1   None     None  

C:\Users\harry\AppData\Local\Temp\ipykernel_12464\3315257287.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


In [ ]:
def insert_data(df):
    conn = get_connection()

    try:
        cursor = conn.cursor()
        for index, row in df.iterrows():
            cursor.execute(
                "INSERT INTO Orders (Column1, Column2) VALUES (%s, %s)",
                (row['Column1'], row['Column2'])
            )
        conn.commit()
    finally:
        conn.close()